# llmEngine Test

This notebook runs **inside the `i2b2-jupyter` Docker container** and calls the actual `llmEngine` and `llmHelper` functions directly — the same code the `jobWatcher` calls when a job is submitted.

Each cell corresponds to one step of the engine pipeline so you can inspect inputs and outputs at every stage.

---

**How to open this notebook:**
1. Start the Jupyter container:  
   `docker compose -f deployment/pg/docker-compose.yml up i2b2-jupyter -d`
2. Open [http://localhost:8888](http://localhost:8888) — token: **`mytoken`**
3. Navigate to `docs/llmEngine_tutorial.ipynb`
4. Run cells top-to-bottom with **Shift+Enter**

---
## Step 0 — Configuration

Edit these values to point at your clinical notes and vLLM endpoint.

In [1]:

NOTE_CONCEPT_PATH   = '/i2b2/Notes/ClinicalNotes/'   
OUTPUT_CONCEPT_PATH = '/i2b2/LLM/NLP/Extract/'       
OUTPUT_CONCEPT_CODE = 'LLM:NLP:FOUND'                


SEARCH_TERMS = ['metformin', 'HbA1c', 'insulin']


VLLM_BASE_URL = 'https://prabinrs--vllm-gemma4-e2b-serve.modal.run'
VLLM_MODEL    = 'gemma4-e2b'

print('Configuration ready.')
print(f'  Note path   : {NOTE_CONCEPT_PATH}')
print(f'  Search terms: {SEARCH_TERMS}')
print(f'  vLLM model  : {VLLM_MODEL}')

Configuration ready.
  Note path   : /i2b2/Notes/ClinicalNotes/
  Search terms: ['metformin', 'HbA1c', 'insulin']
  vLLM model  : gemma4-e2b


---
## Step 1 — Import Engine Modules

These are the exact same imports used inside `llmEngine.py`.

In [2]:
import sys
sys.path.insert(0, '/usr/src/app')   # ensure project root is on path

from i2b2_cdi.config.config import Config
from i2b2_cdi.ML.llmHelper import get_notes, extract_presence, ensure_concepts
from i2b2_cdi.ML.llmEngine import llmEngine
import pandas as pd
from loguru import logger

print('Imports OK.')
print('  Config, get_notes, extract_presence, ensure_concepts, llmEngine — all loaded.')

Imports OK.
  Config, get_notes, extract_presence, ensure_concepts, llmEngine — all loaded.


---
## Step 2 — Build the Config Object

`Config().new_config(argv=['project', 'add'])` reads all DB connection details from the environment variables already set in the container (CRC_DB_HOST, CRC_DB_USER, etc.).

This is identical to what `llmEngine.run()` does internally.

In [3]:
config = Config().new_config(argv=['project', 'add', '-c', '../etl-pgsql.env'])

print('Config built from environment.')
print(f'  DB host : {config.crc_db_host}')
print(f'  DB name : {config.crc_db_name}')
print(f'  DB type : {config.crc_db_type}')

Config built from environment.
  DB host : i2b2-pg
  DB name : i2b2demodata
  DB type : pg


---
## Step 3 — Fetch Clinical Notes (`get_notes`)

`get_notes()` queries `observation_fact JOIN concept_dimension` to find all facts with a non-empty `observation_blob` under the given concept path.

This is **exactly** what happens on the first real step inside `llmEngine.run()`.

In [4]:
from i2b2_cdi.database.cdi_database_connections import I2b2crcDataSource
from i2b2_cdi.database.cdi_db_executor import getDataFrameInChunksUsingCursor
query = """
    SELECT *
    FROM   observation_fact o"""



In [5]:
import pandas as pd 
pd.read_sql(query,crc_ds.connection)

NameError: name 'crc_ds' is not defined

In [8]:
NOTE_CONCEPT_PATH = '/i2b2/Notes/ClinicalNotes/' 

path_pattern = NOTE_CONCEPT_PATH.rstrip('/') + '/%'

In [9]:
with crc_ds as cursor:
    try:
        df = getDataFrameInChunksUsingCursor(
            cursor, query, path_pattern
        )
        logger.info('Fetched {} clinical notes for path: {}', len(df), note_concept_path)
        df.to_dict(orient='records')
    except Exception as e:
        logger.exception('Error fetching clinical notes: {}', e)

NameError: name 'crc_ds' is not defined

In [7]:
notes = get_notes(config, NOTE_CONCEPT_PATH)

print(f'Fetched {len(notes)} note(s).\n')

if notes:
    df_notes = pd.DataFrame(notes)
    df_notes['note_preview'] = df_notes['note_text'].str[:120] + '...'
    display(df_notes[['patient_num', 'encounter_num', 'start_date', 'note_preview']])
else:
    print('No notes found. Check NOTE_CONCEPT_PATH and that observation_blob is populated.')

/usr/src/app/i2b2_cdi/database/cdi_db_executor.py:71: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  for chunk in pd.read_sql_query(sql, cursor.connection, params = {'params': params} ,chunksize=chunksize):
2026-07-09 18:41:20.469 | INFO     | i2b2_cdi.ML.llmHelper:get_notes:32 - Fetched 0 clinical notes for path: /i2b2/Notes/ClinicalNotes/


Fetched 0 note(s).

No notes found. Check NOTE_CONCEPT_PATH and that observation_blob is populated.


---
## Step 4 — Run LLM Extraction (`extract_presence`)

For each note and each search term, `extract_presence()` calls the vLLM chat completions endpoint with a yes/no prompt.

Returns `{patient_num: {term: True/False, ...}, ...}` — aggregated per patient across all their notes.

In [10]:
results = extract_presence(notes, SEARCH_TERMS, VLLM_BASE_URL, VLLM_MODEL)

print('Raw extraction results:')
print(results)
print()

# Pretty display
rows = []
for patient_num, term_results in results.items():
    for term, found in term_results.items():
        rows.append({'patient_num': patient_num, 'term': term, 'found': found})

df_results = pd.DataFrame(rows)
df_results['result'] = df_results['found'].map({True: 'YES', False: 'NO'})
display(df_results[['patient_num', 'term', 'result']])

2026-06-18 02:42:25.993 | WARNING  | i2b2_cdi.ML.llmHelper:extract_presence:88 - vLLM call failed for patient 99001 term metformin: HTTPSConnectionPool(host='prabinrs--vllm-gemma4-e2b-serve.modal.run', port=443): Read timed out. (read timeout=30)
2026-06-18 02:42:56.120 | WARNING  | i2b2_cdi.ML.llmHelper:extract_presence:88 - vLLM call failed for patient 99001 term HbA1c: HTTPSConnectionPool(host='prabinrs--vllm-gemma4-e2b-serve.modal.run', port=443): Read timed out. (read timeout=30)
2026-06-18 02:43:26.341 | WARNING  | i2b2_cdi.ML.llmHelper:extract_presence:88 - vLLM call failed for patient 99001 term insulin: HTTPSConnectionPool(host='prabinrs--vllm-gemma4-e2b-serve.modal.run', port=443): Read timed out. (read timeout=30)
2026-06-18 02:43:26.343 | INFO     | i2b2_cdi.ML.llmHelper:extract_presence:93 - Extraction complete: 1 patients, 3 terms


Raw extraction results:
{99001: {'metformin': False, 'HbA1c': False, 'insulin': False}}



,patient_num,term,result
0,99001,metformin,❌ NO
1,99001,HbA1c,❌ NO
2,99001,insulin,❌ NO


---
## Step 5 — Ensure Concept Codes Exist (`ensure_concepts`)

The i2b2 fact loader validates that every `concept_cd` in the CSV exists in `concept_dimension`.
`ensure_concepts()` auto-inserts child concept codes (e.g. `LLM:NLP:FOUND:metformin`) if they don't already exist.

In [11]:
ensure_concepts(config, OUTPUT_CONCEPT_CODE, SEARCH_TERMS, OUTPUT_CONCEPT_PATH)

# Verify they now exist in the DB
import psycopg2, psycopg2.extras, os

conn = psycopg2.connect(
    host=os.environ['CRC_DB_HOST'],
    port=int(os.environ.get('CRC_DB_PORT', 5432)),
    dbname=os.environ['CRC_DB_NAME'],
    user=os.environ['CRC_DB_USER'],
    password=os.environ['CRC_DB_PASS'],
)
conn.autocommit = True

with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute(f"SET search_path = {os.environ['CRC_DB_NAME']};")
    cur.execute("SELECT concept_cd, name_char FROM concept_dimension WHERE concept_cd LIKE %s ORDER BY concept_cd",
                (OUTPUT_CONCEPT_CODE + ':%',))
    concepts = cur.fetchall()

print(f'Child concepts in concept_dimension ({len(concepts)}):')
for c in concepts:
    print(f"  {c['concept_cd']:<35} {c['name_char']}")

2026-06-18 02:43:44.913 | INFO     | i2b2_cdi.ML.llmHelper:ensure_concepts:134 - Ensured 3 child concepts under LLM:NLP:FOUND


OperationalError: connection to server at "i2b2-pg" (172.19.0.2), port 5432 failed: FATAL:  database "i2b2demodata" does not exist


---
## Step 6 — Build the Facts DataFrame

This is the exact DataFrame construction from `llmEngine.run()` — one row per patient × term.

In [13]:
fact_rows = []
for patient_num, term_results in results.items():
    for term, found in term_results.items():
        fact_rows.append({
            'mrn':        patient_num,
            'code':       '{}/{}'.format(OUTPUT_CONCEPT_CODE, term.replace(' ', '_')),
            'start-date': '1970-01-01 00:00:00',
            'value':      '1' if found else '0',
        })

df_facts = pd.DataFrame(fact_rows)
print('Facts DataFrame to be loaded into i2b2:')
display(df_facts)

Facts DataFrame to be loaded into i2b2:


,mrn,code,start-date,value
0,99001,LLM:NLP:FOUND/metformin,1970-01-01 00:00:00,0
1,99001,LLM:NLP:FOUND/HbA1c,1970-01-01 00:00:00,0
2,99001,LLM:NLP:FOUND/insulin,1970-01-01 00:00:00,0


---
## Step 7 — Load Facts into i2b2 (`send_facts`)

`send_facts()` writes the DataFrame to a CSV, then runs the Mozilla fact loader pipeline which bulk-uploads into `observation_fact`.

In [ ]:
engine = llmEngine(jobId=None)
engine.send_facts(df_facts)

print('send_facts() completed.')

---
## Step 8 — Verify Facts in `observation_fact`

In [ ]:
with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute(f"SET search_path = {os.environ['CRC_DB_NAME']};")
    cur.execute("""
        SELECT patient_num, concept_cd, tval_char AS value, import_date
        FROM   observation_fact
        WHERE  concept_cd LIKE %s
        ORDER BY concept_cd
    """, (OUTPUT_CONCEPT_CODE + ':%',))
    facts_in_db = cur.fetchall()

print(f'Facts in observation_fact ({len(facts_in_db)} rows):\n')
for f in facts_in_db:
    marker = 'FOUND' if f['value'] == '1' else ' NOT FOUND'
    term = f['concept_cd'].split(':')[-1]
    print(f"  patient={f['patient_num']}  {term:<12}  {marker}")

conn.close()

---
## Summary

You just ran the full `llmEngine` pipeline step by step:

| Function | What it did |
|----------|-------------|
| `Config().new_config()` | Read DB credentials from container env vars |
| `get_notes()` | Fetched clinical notes from `observation_fact.observation_blob` |
| `extract_presence()` | Called vLLM for each note × term → boolean result |
| `ensure_concepts()` | Auto-created child concept codes in `concept_dimension` |


When `jobWatcher` runs `llmEngine`, it executes exactly these steps in `llmEngine.run()` — this notebook just lets you see each one individually.